# Hafta 5: Gradio ile Model Dağıtımı (Deployment)

Bu defterde eğittiğimiz araç fiyat tahmin modelini **Gradio** kütüphanesi ile web arayüzüne dönüştüreceğiz.

**İçerik:**
- Gradio kurulumu
- Modeli yükleme veya hızlı eğitim
- Gradio arayüzü oluşturma
- Yerel demo çalıştırma
- Hugging Face Spaces'e dağıtım adımları

## 1. Gradio Kurulumu

### Gerekli Paketlerin Kurulumu

Aşağıdaki komut ile ihtiyaç duyulan Python paketlerini yüklüyoruz.

In [ ]:
!pip install gradio -q

### Kütüphanelerin Yüklenmesi

Projede kullanacağımız kütüphaneleri içe aktarıyoruz:

| Kütüphane | Amacı |
|-----------|-------|
| `gradio` | ML modelleri için web arayüzü oluşturma |
| `joblib` | Model kaydetme/yükleme |
| `numpy` | Sayısal hesaplamalar ve dizi işlemleri |
| `pandas` | Veri çerçeveleri (DataFrame) ile veri analizi |
| `sklearn` | Makine öğrenmesi algoritmaları ve araçları |
| `warnings` | Uyarı mesajlarını yönetme |


In [ ]:
import numpy as np
import pandas as pd
import gradio as gr
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings('ignore')

print(f"Gradio sürümü: {gr.__version__}")
print("Kütüphaneler başarıyla yüklendi!")

## 2. Modeli Yükleme veya Hızlı Eğitim

Önceki defterde kaydettiğimiz modeli yüklemeye çalışacağız. Eğer dosya yoksa hızlıca yeniden eğiteceğiz.

In [ ]:
import os

MODEL_DOSYASI = 'arac_fiyat_modeli.pkl'

if os.path.exists(MODEL_DOSYASI):
    model_data = joblib.load(MODEL_DOSYASI)
    print(f"Model '{MODEL_DOSYASI}' dosyasından yüklendi!")
else:
    print("Model dosyası bulunamadı. Hızlı eğitim başlıyor...")
    
    # Hızlı veri oluşturma ve eğitim
    np.random.seed(42)
    n = 500
    
    markalar = ['Toyota', 'Honda', 'Volkswagen', 'BMW', 'Mercedes', 'Renault', 'Fiat', 'Hyundai', 'Ford', 'Audi']
    marka_baz = {'Toyota': 450000, 'Honda': 420000, 'Volkswagen': 480000, 'BMW': 750000,
                 'Mercedes': 800000, 'Renault': 320000, 'Fiat': 280000, 'Hyundai': 380000,
                 'Ford': 360000, 'Audi': 700000}
    yakit_tipleri = ['Benzin', 'Dizel', 'Hibrit', 'LPG']
    vites_tipleri = ['Manuel', 'Otomatik']
    
    data = []
    for _ in range(n):
        marka = np.random.choice(markalar)
        model_yili = np.random.randint(2010, 2025)
        km = np.random.randint(0, 250000)
        yakit = np.random.choice(yakit_tipleri, p=[0.35, 0.35, 0.1, 0.2])
        vites = np.random.choice(vites_tipleri, p=[0.45, 0.55])
        motor = np.random.choice([1.0, 1.2, 1.4, 1.5, 1.6, 1.8, 2.0, 2.5, 3.0])
        
        baz = marka_baz[marka]
        fiyat = max(50000, baz + (model_yili-2010)*35000 - km*1.2 +
                    {'Benzin':0,'Dizel':30000,'Hibrit':80000,'LPG':-20000}[yakit] +
                    (50000 if vites=='Otomatik' else 0) + motor*40000 +
                    np.random.normal(0, 30000))
        
        data.append({'Marka': marka, 'Model_Yili': model_yili, 'Kilometre': km,
                     'Yakit_Tipi': yakit, 'Vites': vites, 'Motor_Hacmi': motor,
                     'Fiyat': round(fiyat, -3)})
    
    df = pd.DataFrame(data)
    df_enc = df.copy()
    
    label_encoders = {}
    for col in ['Marka', 'Yakit_Tipi', 'Vites']:
        le = LabelEncoder()
        df_enc[col] = le.fit_transform(df_enc[col])
        label_encoders[col] = le
    
    X = df_enc.drop('Fiyat', axis=1)
    y = df_enc['Fiyat']
    
    model = LinearRegression()
    model.fit(X, y)
    
    model_data = {'model': model, 'label_encoders': label_encoders, 'features': list(X.columns)}
    joblib.dump(model_data, MODEL_DOSYASI)
    print(f"Model eğitildi ve '{MODEL_DOSYASI}' olarak kaydedildi!")

print(f"\nModel özellikleri: {model_data['features']}")
print(f"Encoder'lar: {list(model_data['label_encoders'].keys())}")

## 3. Tahmin Fonksiyonu Oluşturma

In [ ]:
def arac_fiyat_tahmin(marka, model_yili, kilometre, yakit_tipi, vites, motor_hacmi):
    """
    Araç özelliklerine göre fiyat tahmini yapan fonksiyon.
    """
    try:
        # Giriş verilerini hazırla
        girdi = pd.DataFrame([{
            'Marka': marka,
            'Model_Yili': int(model_yili),
            'Kilometre': int(kilometre),
            'Yakit_Tipi': yakit_tipi,
            'Vites': vites,
            'Motor_Hacmi': float(motor_hacmi)
        }])
        
        # Kategorik değişkenleri encode et
        for col in ['Marka', 'Yakit_Tipi', 'Vites']:
            le = model_data['label_encoders'][col]
            if girdi[col].values[0] not in le.classes_:
                return f"Hata: '{girdi[col].values[0]}' değeri tanınmıyor!"
            girdi[col] = le.transform(girdi[col])
        
        # Tahmin yap
        tahmin = model_data['model'].predict(girdi)[0]
        tahmin = max(0, tahmin)  # Negatif fiyat olmasın
        
        return f"Tahmini Fiyat: {tahmin:,.0f} TL"
    
    except Exception as e:
        return f"Hata oluştu: {str(e)}"

# Test edelim
sonuc = arac_fiyat_tahmin('BMW', 2021, 45000, 'Benzin', 'Otomatik', 2.0)
print(f"Test sonucu: {sonuc}")

## 4. Gradio Arayüzü Oluşturma

Gradio, makine öğrenmesi modellerini hızlıca web arayüzüne dönüştürmemizi sağlar.

In [ ]:
# Marka listesi (encoder'dan al)
markalar = list(model_data['label_encoders']['Marka'].classes_)
yakit_tipleri = list(model_data['label_encoders']['Yakit_Tipi'].classes_)
vites_tipleri = list(model_data['label_encoders']['Vites'].classes_)

print(f"Markalar: {markalar}")
print(f"Yakıt tipleri: {yakit_tipleri}")
print(f"Vites tipleri: {vites_tipleri}")

### Gradio Arayüzü Oluşturma

Eğitilmiş modelimiz için kullanıcı dostu bir web arayüzü oluşturuyoruz. Gradio ile slider, dropdown gibi giriş bileşenleri tanımlayıp modeli herkesin kullanabileceği hale getiriyoruz.

In [ ]:
# Gradio arayüzü
demo = gr.Interface(
    fn=arac_fiyat_tahmin,
    inputs=[
        gr.Dropdown(
            choices=markalar,
            value='Toyota',
            label='Marka'
        ),
        gr.Slider(
            minimum=2010,
            maximum=2025,
            step=1,
            value=2020,
            label='Model Yılı'
        ),
        gr.Slider(
            minimum=0,
            maximum=300000,
            step=5000,
            value=50000,
            label='Kilometre'
        ),
        gr.Dropdown(
            choices=yakit_tipleri,
            value='Benzin',
            label='Yakıt Tipi'
        ),
        gr.Dropdown(
            choices=vites_tipleri,
            value='Otomatik',
            label='Vites'
        ),
        gr.Dropdown(
            choices=[1.0, 1.2, 1.4, 1.5, 1.6, 1.8, 2.0, 2.5, 3.0],
            value=1.6,
            label='Motor Hacmi (L)'
        ),
    ],
    outputs=gr.Textbox(label='Tahmin Sonucu'),
    title='Araç Fiyat Tahmin Sistemi',
    description='İkinci el araç özelliklerini girerek tahmini fiyatı öğrenin. Model, lineer regresyon ile eğitilmiştir.',
    examples=[
        ['BMW', 2021, 45000, 'Benzin', 'Otomatik', 2.0],
        ['Fiat', 2018, 120000, 'LPG', 'Manuel', 1.4],
        ['Mercedes', 2023, 15000, 'Dizel', 'Otomatik', 2.0],
        ['Renault', 2016, 180000, 'Benzin', 'Manuel', 1.2],
        ['Toyota', 2022, 30000, 'Hibrit', 'Otomatik', 1.8],
    ],
    theme=gr.themes.Soft()
)

print("Gradio arayüzü hazır!")

## 5. Yerel Demo Çalıştırma

### Yerel sunucuda çalıştırma

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# Yerel sunucuda çalıştırma
# share=True parametresi ile geçici bir public URL de oluşturulabilir
demo.launch(share=False)

## 6. Hugging Face Spaces'e Dağıtım

Modelinizi dünya ile paylaşmak için Hugging Face Spaces kullanabilirsiniz. Aşağıda adım adım talimatlar verilmiştir.

### Adım 1: Hugging Face Hesabı Oluşturma

1. [huggingface.co](https://huggingface.co) adresine gidin
2. Sağ üstteki **Sign Up** butonuna tıklayın
3. E-posta, kullanıcı adı ve şifre ile kayıt olun
4. E-posta doğrulamasını tamamlayın

### Adım 2: Yeni Space Oluşturma

1. Giriş yaptıktan sonra [huggingface.co/new-space](https://huggingface.co/new-space) adresine gidin
2. **Space name**: `arac-fiyat-tahmini` (veya istediğiniz bir isim)
3. **License**: MIT veya Apache 2.0
4. **SDK**: **Gradio** seçin
5. **Hardware**: CPU Basic (ücretsiz)
6. **Create Space** butonuna tıklayın

### Adım 3: Gerekli Dosyaları Hazırlama

Space'e yüklenecek 3 dosyaya ihtiyacınız var:

1. **`app.py`** - Ana uygulama dosyası
2. **`requirements.txt`** - Bağımlılıklar
3. **`arac_fiyat_modeli.pkl`** - Eğitilmiş model

In [ ]:
# app.py dosyasını oluşturalım
app_kodu = '''
import numpy as np
import pandas as pd
import gradio as gr
import joblib
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder

# Modeli yükle
model_data = joblib.load("arac_fiyat_modeli.pkl")

markalar = list(model_data["label_encoders"]["Marka"].classes_)
yakit_tipleri = list(model_data["label_encoders"]["Yakit_Tipi"].classes_)
vites_tipleri = list(model_data["label_encoders"]["Vites"].classes_)

def arac_fiyat_tahmin(marka, model_yili, kilometre, yakit_tipi, vites, motor_hacmi):
    try:
        girdi = pd.DataFrame([{
            "Marka": marka,
            "Model_Yili": int(model_yili),
            "Kilometre": int(kilometre),
            "Yakit_Tipi": yakit_tipi,
            "Vites": vites,
            "Motor_Hacmi": float(motor_hacmi)
        }])
        
        for col in ["Marka", "Yakit_Tipi", "Vites"]:
            le = model_data["label_encoders"][col]
            girdi[col] = le.transform(girdi[col])
        
        tahmin = model_data["model"].predict(girdi)[0]
        tahmin = max(0, tahmin)
        
        return f"Tahmini Fiyat: {tahmin:,.0f} TL"
    except Exception as e:
        return f"Hata: {str(e)}"

demo = gr.Interface(
    fn=arac_fiyat_tahmin,
    inputs=[
        gr.Dropdown(choices=markalar, value="Toyota", label="Marka"),
        gr.Slider(minimum=2010, maximum=2025, step=1, value=2020, label="Model Yılı"),
        gr.Slider(minimum=0, maximum=300000, step=5000, value=50000, label="Kilometre"),
        gr.Dropdown(choices=yakit_tipleri, value="Benzin", label="Yakıt Tipi"),
        gr.Dropdown(choices=vites_tipleri, value="Otomatik", label="Vites"),
        gr.Dropdown(choices=[1.0, 1.2, 1.4, 1.5, 1.6, 1.8, 2.0, 2.5, 3.0], value=1.6, label="Motor Hacmi (L)"),
    ],
    outputs=gr.Textbox(label="Tahmin Sonucu"),
    title="Araç Fiyat Tahmin Sistemi",
    description="İkinci el araç özelliklerini girerek tahmini fiyatı öğrenin.",
    examples=[
        ["BMW", 2021, 45000, "Benzin", "Otomatik", 2.0],
        ["Fiat", 2018, 120000, "LPG", "Manuel", 1.4],
        ["Mercedes", 2023, 15000, "Dizel", "Otomatik", 2.0],
    ],
    theme=gr.themes.Soft()
)

demo.launch()
'''

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(app_kodu.strip())

print("app.py dosyası oluşturuldu!")

### Model Kaydetme

Eğitilmiş modeli diske kaydediyoruz. Böylece modeli tekrar eğitmeden doğrudan yükleyip tahmin yapabiliriz.

In [ ]:
# requirements.txt dosyasını oluşturalım
requirements = """numpy
pandas
scikit-learn
gradio
joblib
"""

with open('requirements.txt', 'w') as f:
    f.write(requirements.strip())

print("requirements.txt dosyası oluşturuldu!")

### Adım 4: Dosyaları Hugging Face Space'e Yükleme

#### Yöntem 1: Web Arayüzü ile (Kolay)

1. Oluşturduğunuz Space sayfasına gidin: `https://huggingface.co/spaces/KULLANICI_ADINIZ/arac-fiyat-tahmini`
2. **Files and versions** sekmesine tıklayın
3. **Add file** > **Upload files** butonuna tıklayın
4. Aşağıdaki 3 dosyayı sürükleyip bırakın:
   - `app.py`
   - `requirements.txt`
   - `arac_fiyat_modeli.pkl`
5. **Commit changes** butonuna tıklayın

#### Yöntem 2: Git ile (İleri Düzey)

```bash
# Space'i klonlayın
git clone https://huggingface.co/spaces/KULLANICI_ADINIZ/arac-fiyat-tahmini
cd arac-fiyat-tahmini

# Dosyaları kopyalayın
cp /path/to/app.py .
cp /path/to/requirements.txt .
cp /path/to/arac_fiyat_modeli.pkl .

# Commit ve push
git add .
git commit -m "İlk sürüm: Araç fiyat tahmin uygulaması"
git push
```

### Adım 5: Uygulamayı Test Etme

1. Dosyaları yükledikten sonra **Build** otomatik başlar
2. Build süreci 2-5 dakika sürebilir (bağımlılıkların yüklenmesi)
3. Build tamamlandığında **App** sekmesinde uygulamanız görünecek
4. URL formatı: `https://huggingface.co/spaces/KULLANICI_ADINIZ/arac-fiyat-tahmini`

Bu URL'yi herkesle paylaşabilirsiniz!

### Adım 6: Sorun Giderme

Eğer uygulama çalışmazsa:

1. **Logs** sekmesine tıklayarak hata mesajlarını kontrol edin
2. En sık karşılaşılan sorunlar:
   - `ModuleNotFoundError`: `requirements.txt`'e eksik kütüphane ekleyin
   - `FileNotFoundError`: Model dosyasının yüklendiğinden emin olun
   - `sklearn version mismatch`: Lokal ve Space'deki sklearn sürümleri farklı olabilir

3. Sklearn sürüm uyumsuzluğu için `requirements.txt`'e sürüm ekleyin:
   ```
   scikit-learn==1.3.0
   ```

## 7. Gradio Blocks ile Gelişmiş Arayüz (Bonus)

`gr.Interface` yerine `gr.Blocks` kullanarak daha özelleştirilmiş arayüzler oluşturabilirsiniz.

In [ ]:
# Gelişmiş arayüz örneği
with gr.Blocks(theme=gr.themes.Soft(), title="Araç Fiyat Tahmini") as demo_blocks:
    gr.Markdown("# Araç Fiyat Tahmin Sistemi")
    gr.Markdown("İkinci el araç bilgilerini girerek tahmini fiyatı öğrenin.")
    
    with gr.Row():
        with gr.Column():
            gr.Markdown("### Araç Bilgileri")
            marka_input = gr.Dropdown(choices=markalar, value='Toyota', label='Marka')
            yil_input = gr.Slider(minimum=2010, maximum=2025, step=1, value=2020, label='Model Yılı')
            km_input = gr.Slider(minimum=0, maximum=300000, step=5000, value=50000, label='Kilometre')
        
        with gr.Column():
            gr.Markdown("### Teknik Özellikler")
            yakit_input = gr.Dropdown(choices=yakit_tipleri, value='Benzin', label='Yakıt Tipi')
            vites_input = gr.Dropdown(choices=vites_tipleri, value='Otomatik', label='Vites')
            motor_input = gr.Dropdown(
                choices=[1.0, 1.2, 1.4, 1.5, 1.6, 1.8, 2.0, 2.5, 3.0],
                value=1.6, label='Motor Hacmi (L)'
            )
    
    tahmin_btn = gr.Button("Fiyat Tahmin Et", variant="primary", size="lg")
    sonuc_output = gr.Textbox(label="Tahmin Sonucu", lines=2)
    
    tahmin_btn.click(
        fn=arac_fiyat_tahmin,
        inputs=[marka_input, yil_input, km_input, yakit_input, vites_input, motor_input],
        outputs=sonuc_output
    )
    
    gr.Markdown("---")
    gr.Markdown("*Bu model lineer regresyon ile eğitilmiştir. Tahminler yaklaşık değerlerdir.*")

print("Gelişmiş arayüz hazır!")

### Gelişmiş arayüzü çalıştırma

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# Gelişmiş arayüzü çalıştırma
demo_blocks.launch(share=False)

## 8. Özet

Bu defterde öğrendiklerimiz:

1. **Gradio** kütüphanesini kurduk ve temellerini öğrendik
2. Eğitilmiş modeli **joblib** ile yükledik
3. `gr.Interface` ile basit bir **web arayüzü** oluşturduk
4. Slider, Dropdown gibi **giriş bileşenleri** kullandık
5. `gr.Blocks` ile daha **gelişmiş bir arayüz** tasarladık
6. **Hugging Face Spaces**'e dağıtım adımlarını öğrendik

### Önemli Noktalar
- Gradio, modelleri hızlıca web arayüzüne dönüştürür
- `share=True` ile geçici public URL oluşturulabilir
- Kalıcı dağıtım için Hugging Face Spaces kullanılır
- `gr.Blocks` ile daha özelleştirilmiş arayüzler mümkündür